# Phase 4 — Per-Layer CGRS (Curvature-Guided Rank Selection)
### ViT-Base/16 · CIFAR-100 · Google Colab T4

**What this notebook does:**
1. Mounts Google Drive and loads all Phase 1–3 results
2. Implements **Per-Layer CGRS** — 12 independent LoRA adapters (one per transformer block), each with its own dynamically adjusted rank driven by its own per-layer λ_max
3. Runs **3 experiments** with a shared τ sweep calibrated from Phase 2 per-layer λ_max data
4. Computes full comparison table: Per-Layer CGRS vs Global CGRS (Phase 3) vs Fixed-rank baselines
5. Generates all publication-quality plots and saves results to JSON

**Key research claim to validate:**  
Per-Layer CGRS allocates higher rank to sharp deep layers (L9–L11 Query) and lower rank to flat early layers (L0–L3), achieving better accuracy-per-parameter than both fixed-rank LoRA *and* global CGRS (Phase 3).


## Cell 1 · Install Dependencies & Imports

In [1]:
# ── Cell 1: Verify CARC Environment ─────────────────────────────────────────
import sys, os, torch, transformers, peft

print(f"Python:        {sys.version}")
print(f"PyTorch:       {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:           {torch.cuda.get_device_name(0)}")
    print(f"VRAM:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected — check your salloc request.")
print(f"Transformers:  {transformers.__version__}")
print(f"PEFT:          {peft.__version__}")

/home1/jkshaikh/.conda/envs/cgrs_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python:        3.11.15 | packaged by conda-forge | (main, Mar  5 2026, 16:45:40) [GCC 14.3.0]
PyTorch:       2.2.0+cu121
CUDA available: True
GPU:           Tesla V100-PCIE-32GB
VRAM:          34.1 GB
Transformers:  4.40.0
PEFT:          0.10.0


## Cell 2 — Paths & Load Previous Results

In [2]:
# ── Cell 2: CARC Local Paths & Load Previous Results ────────────────────────
import os, json
import numpy as np

PROJECT_BASE = os.path.expanduser("~/CGRS_Project/CGRS_Buildup")   # ← keep your path

PHASE1_DIR  = os.path.join(PROJECT_BASE, "phase1_results")
PHASE2_DIR  = os.path.join(PROJECT_BASE, "phase2_results")
PHASE3_DIR  = os.path.join(PROJECT_BASE, "phase3_results")
PHASE4_DIR  = os.path.join(PROJECT_BASE, "phase4_results")
PLOTS_DIR   = os.path.join(PROJECT_BASE, "plots")
DATA_DIR    = os.path.join(PROJECT_BASE, "data")

for d in [PHASE1_DIR, PHASE2_DIR, PHASE3_DIR, PHASE4_DIR, PLOTS_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"PROJECT_BASE: {PROJECT_BASE}")

# ── Load Phase 1 config & results ─────────────────────────────────────────────
config_path  = os.path.join(PHASE1_DIR, "config.json")
results_path = os.path.join(PHASE1_DIR, "results.json")

if not os.path.exists(config_path):
    raise FileNotFoundError(f"config.json not found at {config_path}")

with open(config_path) as f:
    CONFIG = json.load(f)
with open(results_path) as f:
    phase1_results = json.load(f)

print(f"\n✓ Phase 1 config loaded")
for k, v in phase1_results.items():
    print(f"  {k:25s} Test Acc: {v['test_acc']:.2f}%")

# ── Load Phase 2 per-layer λ_max for τ calibration ────────────────────────────
ALL_RESULTS_PATH = os.path.join(PHASE2_DIR, "all_results_complete.json")
PHASE2_LAYER_LMAX_R16 = {}
pct_50 = 5e-5  # fallback

if os.path.exists(ALL_RESULTS_PATH):
    with open(ALL_RESULTS_PATH) as f:
        all_results_p2 = json.load(f)

    # ✅ Correct key is "r=16", not "r16"
    curvature_data = all_results_p2.get("curvature", {})
    r16_block      = curvature_data.get("r=16", {})
    r16_per_layer  = r16_block.get("per_layer", {})

    if r16_per_layer:
        for key_str, stats in r16_per_layer.items():
            try:
                # key_str looks like "(0, 'query')" — eval it safely
                key = eval(key_str)
                PHASE2_LAYER_LMAX_R16[key] = stats.get("lambda_max", 0.0)
            except Exception:
                pass  # skip malformed keys

        if PHASE2_LAYER_LMAX_R16:
            vals   = list(PHASE2_LAYER_LMAX_R16.values())
            pct_50 = float(np.percentile(vals, 50))
            print(f"\n✓ Phase 2 per-layer λ_max (r=16) loaded: {len(PHASE2_LAYER_LMAX_R16)} layer-proj pairs")
            print(f"  λ_max range: {min(vals):.6f} – {max(vals):.6f}")
            print(f"  τ percentiles → 25th: {np.percentile(vals,25):.6f} | "
                  f"50th: {pct_50:.6f} | 75th: {np.percentile(vals,75):.6f}")
        else:
            print("⚠ r=16 per_layer block was empty after parsing — check key format.")
    else:
        print("⚠ 'r=16' found but 'per_layer' sub-key missing.")
        print(f"  Available curvature keys: {list(curvature_data.keys())[:6]}")
else:
    print(f"⚠ all_results_complete.json not found at {ALL_RESULTS_PATH}")
    print("  Using fallback τ = 5e-5")

PROJECT_BASE: /home1/jkshaikh/CGRS_Project/CGRS_Buildup

✓ Phase 1 config loaded
  Full fine-tune            Test Acc: 92.87%
  Frozen backbone           Test Acc: 86.46%
  LoRA r=3                  Test Acc: 84.77%
  LoRA r=5                  Test Acc: 87.36%
  LoRA r=6                  Test Acc: 87.27%
  LoRA r=10                 Test Acc: 89.02%
  LoRA r=12                 Test Acc: 89.31%
  LoRA r=16                 Test Acc: 89.51%
  LoRA r=30                 Test Acc: 90.30%
  LoRA r=52                 Test Acc: 90.62%
  LoRA r=64                 Test Acc: 90.48%
  LoRA r=80                 Test Acc: 90.42%
  LoRA r=100                Test Acc: 90.90%
  LoRA r=128                Test Acc: 90.55%
  LoRA r=150                Test Acc: 90.70%
  LoRA r=200                Test Acc: 90.95%

✓ Phase 2 per-layer λ_max (r=16) loaded: 24 layer-proj pairs
  λ_max range: 0.000034 – 0.012323
  τ percentiles → 25th: 0.000242 | 50th: 0.000486 | 75th: 0.001595


## Cell 3 — Imports & Reproducibility

In [3]:
# ── Cell 3: Imports & Reproducibility ───────────────────────────────────────
import random, time, json, os
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from transformers import ViTForImageClassification, ViTImageProcessor
from peft import LoraConfig, get_peft_model
from scipy import stats

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — training will be very slow.")

Device: cuda
GPU:    Tesla V100-PCIE-32GB
VRAM:   34.1 GB


## Cell 4 — Dataset & DataLoaders

In [4]:
# ── Cell 4: Dataset & DataLoaders ───────────────────────────────────────────
try:
    processor = ViTImageProcessor.from_pretrained(CONFIG["model_name"])
    mean = processor.image_mean
    std  = processor.image_std
    print(f"Processor loaded: {CONFIG['model_name']}")
except Exception:
    mean = [0.5, 0.5, 0.5]
    std  = [0.5, 0.5, 0.5]
    print("Using default ViT-Base normalization.")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

print("Downloading/loading CIFAR-100...")
full_train = datasets.CIFAR100(root=DATA_DIR, train=True,  download=True, transform=transform)
test_dataset = datasets.CIFAR100(root=DATA_DIR, train=False, download=True, transform=transform)

val_size = len(full_train) - CONFIG["train_size"]
train_dataset, val_dataset = random_split(
    full_train,
    [CONFIG["train_size"], val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# V100 has 32GB VRAM — can use more workers safely
NUM_WORKERS = 4

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")
print(f"Test:  {len(test_dataset):,}")

Processor loaded: google/vit-base-patch16-224
Downloading/loading CIFAR-100...


/home1/jkshaikh/.conda/envs/cgrs_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Files already downloaded and verified
Files already downloaded and verified
Train: 45,000
Val:   5,000
Test:  10,000


## Cell 5 — Phase 4 Hyperparameters

In [5]:
# ── Cell 5: Phase 4 Hyperparameters ─────────────────────────────────────────
P4 = {
    "r_init":        3,
    "r_min":         3,
    "r_max":         64,
    "rank_list":     [3, 5, 6, 10, 12, 16, 30, 52, 64],
    "check_every":   200,
    "cooldown":      100,
    "probe_batches": 8,
    "epochs":        3,
    "lr":            5e-4,
    "weight_decay":  0.01,
    "lora_alpha":    16,
}

all_lmax_r16 = list(PHASE2_LAYER_LMAX_R16.values()) if PHASE2_LAYER_LMAX_R16 else [5e-5]
pct_25 = float(np.percentile(all_lmax_r16, 25))
pct_50 = float(np.percentile(all_lmax_r16, 50))
pct_75 = float(np.percentile(all_lmax_r16, 75))

print("Phase 2 λ_max (r=16) distribution across 24 layer-proj pairs:")
print(f"  Min:  {min(all_lmax_r16):.6f}")
print(f"  25th: {pct_25:.6f}")
print(f"  50th: {pct_50:.6f}")
print(f"  75th: {pct_75:.6f}")
print(f"  Max:  {max(all_lmax_r16):.6f}")

print(f"\nPhase 4 config:")
print(f"  r_init={P4['r_init']} (expand-only: starts at r_min, grows when λ > τ)")
print(f"  r_min={P4['r_min']}, r_max={P4['r_max']}")
print(f"  rank_list={P4['rank_list']}")
print(f"  check_every={P4['check_every']} steps, cooldown={P4['cooldown']}")

Phase 2 λ_max (r=16) distribution across 24 layer-proj pairs:
  Min:  0.000034
  25th: 0.000242
  50th: 0.000486
  75th: 0.001595
  Max:  0.012323

Phase 4 config:
  r_init=3 (expand-only: starts at r_min, grows when λ > τ)
  r_min=3, r_max=64
  rank_list=[3, 5, 6, 10, 12, 16, 30, 52, 64]
  check_every=200 steps, cooldown=100


## Cell 6 — Model Builder & Utilities


In [6]:
# ── Cell 6: Model Builder & Utilities ────────────────────────────────────────
def build_lora_model(r):
    base = ViTForImageClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=CONFIG["num_classes"],
        ignore_mismatched_sizes=True,
    )
    cfg = LoraConfig(
        r=r,
        lora_alpha=P4["lora_alpha"],
        lora_dropout=CONFIG.get("lora_dropout", 0.1),
        target_modules=CONFIG.get("target_modules", ["query", "value"]),
        bias="none",
    )
    return get_peft_model(base, cfg).to(device)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(pixel_values=imgs)
        loss_sum += criterion(out.logits, labels).item()
        correct  += (out.logits.argmax(dim=1) == labels).sum().item()
        total    += labels.size(0)
    return 100.0 * correct / total, loss_sum / len(loader)

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_optimizer(model, lr, weight_decay):
    return optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=weight_decay
    )

def get_scheduler(optimizer, total_steps):
    return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(total_steps, 1))

def next_rank(current, rank_list):
    idx = rank_list.index(current) if current in rank_list else 0
    return rank_list[min(idx + 1, len(rank_list) - 1)]


def transition_lora_rank(old_model, old_r, new_r):
    """SVD-based weight transfer — reads actual tensor shapes, not old_r."""
    old_params = {n: p.data.clone().cpu() for n, p in old_model.named_parameters()}
    new_model = build_lora_model(new_r)
    new_model.cpu()
    new_param_dict = dict(new_model.named_parameters())

    # Copy all non-LoRA weights
    for name, old_data in old_params.items():
        if "lora_A" not in name and "lora_B" not in name:
            if name in new_param_dict:
                new_param_dict[name].data.copy_(old_data)

    # SVD-transfer LoRA weights per adapter
    lora_bases = {}
    for name, data in old_params.items():
        if "lora_A" in name:
            base = name[:name.index("lora_A")]
            lora_bases.setdefault(base, {})["A"] = data
            lora_bases[base]["A_name"] = name
        elif "lora_B" in name:
            base = name[:name.index("lora_B")]
            lora_bases.setdefault(base, {})["B"] = data
            lora_bases[base]["B_name"] = name

    for base, pair in lora_bases.items():
        if "A" not in pair or "B" not in pair:
            continue
        A, B = pair["A"].float(), pair["B"].float()

        # ── FIX: read actual rank from tensor shape, not old_r ──
        actual_old_r = A.shape[0]   # lora_A is [r, in_features]

        if new_r > actual_old_r:
            # Pad with zeros
            new_A = torch.zeros(new_r, A.shape[1])
            new_A[:actual_old_r] = A
            new_B = torch.zeros(B.shape[0], new_r)
            new_B[:, :actual_old_r] = B
        else:
            # Truncate via SVD
            W = B @ A
            try:
                U, S, Vh = torch.linalg.svd(W, full_matrices=False)
                sqrt_S = torch.sqrt(S[:new_r].clamp(min=0.0))
                new_B = U[:, :new_r] * sqrt_S
                new_A = Vh[:new_r] * sqrt_S.unsqueeze(1)
            except RuntimeError:
                new_A = A[:new_r]
                new_B = B[:, :new_r]

        for name, param in new_model.named_parameters():
            if name.startswith(base) and "lora_A" in name:
                param.data.copy_(new_A.to(param.dtype))
            elif name.startswith(base) and "lora_B" in name:
                param.data.copy_(new_B.to(param.dtype))

    return new_model.to(device)

print("Model builder and utilities ready.")

Model builder and utilities ready.


## Cell 7 — PerLayerLoRAViT & Fisher Probe

In [7]:
# ── Cell 7: Per-Layer Fisher Probe using PEFT named parameters ───────────────
def compute_perlayer_fisher(model, loader, n_batches):
    """
    Accumulates diagonal Fisher per transformer layer using PEFT's own
    lora_A / lora_B parameters. These DO flow gradients through forward().
    Returns dict: {(layer_idx, 'query'|'value'): lambda_max}
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()
    fisher_accum = {}   # key -> accumulated grad^2 tensor
    batches_done = 0

    for imgs, labels in loader:
        if batches_done >= n_batches:
            break
        imgs, labels = imgs.to(device), labels.to(device)
        model.zero_grad()
        out  = model(pixel_values=imgs)
        loss = criterion(out.logits, labels)
        loss.backward()

        with torch.no_grad():
            for name, param in model.named_parameters():
                if param.grad is None or not param.requires_grad:
                    continue
                if "lora_A" not in name and "lora_B" not in name:
                    continue
                # PEFT names look like:
                # base_model.model.vit.encoder.layer.{i}.attention.attention.{query/value}.lora_A.default.weight
                parts = name.split(".")
                layer_idx = None
                proj = None
                for j, p in enumerate(parts):
                    if p == "layer" and j + 1 < len(parts):
                        try:
                            layer_idx = int(parts[j + 1])
                        except ValueError:
                            pass
                    if p in ("query", "value"):
                        proj = p
                if layer_idx is None or proj is None:
                    continue

                key = (layer_idx, proj)
                g2  = param.grad.detach().pow(2).flatten()
                if key not in fisher_accum:
                    fisher_accum[key] = g2.clone()
                else:
                    fisher_accum[key] += g2

        batches_done += 1

    model.zero_grad()
    result = {}
    for key, accum in fisher_accum.items():
        if batches_done > 0:
            result[key] = float((accum / batches_done).max().item())
        else:
            result[key] = 0.0

    # Fill missing keys with 0
    for i in range(12):
        for proj in ["query", "value"]:
            result.setdefault((i, proj), 0.0)

    return result


def global_lmax(lmax_dict):
    vals = [v for v in lmax_dict.values() if v > 0]
    return max(vals) if vals else 0.0


def load_if_exists(run_name, save_dir):
    path = os.path.join(save_dir, f"{run_name}_result.json")
    if os.path.exists(path):
        with open(path) as f:
            result = json.load(f)
        print(f"✓ {run_name} already saved — "
              f"Test Acc: {result.get('test_acc', 0):.2f}%, "
              f"Avg Rank: {result.get('avg_rank', 0):.2f} — skipping.")
        return result
    return None


phase4_results = {}
print("Per-layer Fisher probe (PEFT-native) ready.")
print("  Probes lora_A/lora_B gradients in PEFT named params — gradients guaranteed non-zero.")

Per-layer Fisher probe (PEFT-native) ready.
  Probes lora_A/lora_B gradients in PEFT named params — gradients guaranteed non-zero.


## Cell 8 — Training Loop (Expand-Only)

In [8]:
# ── Cell 8: Per-Layer CGRS Training Loop (PEFT-native Fisher) ────────────────
def run_perlayer_cgrs(run_name, tau_config, r_init=16, epochs=3,
                      save_dir=None, expand_only=True):
    """
    Per-layer CGRS using standard PEFT LoRA model.
    tau_config: float (shared) or dict {(layer_idx, proj): tau_val} (per-layer).
    r_init: starting rank for ALL layers. Use 16 to match Phase 3 baseline.
    expand_only: if True, rank only increases (never decreases).
    """
    is_per_layer_tau = isinstance(tau_config, dict)
    tau_display = "per-layer" if is_per_layer_tau else f"{tau_config:.2e}"
    mode = "EXPAND-ONLY" if expand_only else "BIDIRECTIONAL"
    print(f"\n{'='*70}")
    print(f"  Run:    {run_name}")
    print(f"  τ:      {tau_display}")
    print(f"  r_init: {r_init}  [mode: {mode}]")
    print(f"{'='*70}")

    RANK_LIST = P4["rank_list"]
    R_MIN     = P4["r_min"]
    R_MAX     = P4["r_max"]
    K         = P4["check_every"]
    COOLDOWN  = P4["cooldown"]
    N_PROBE   = P4["probe_batches"]

    # Build single PEFT model — r_init applies globally at start
    model      = build_lora_model(r_init)
    optimizer  = get_optimizer(model, lr=P4["lr"], weight_decay=P4["weight_decay"])
    total_steps = epochs * len(train_loader)
    scheduler  = get_scheduler(optimizer, total_steps)
    criterion  = nn.CrossEntropyLoss()

    # Per-layer rank state (all start at r_init)
    layer_ranks      = {i: r_init for i in range(12)}
    cooldown_left    = {i: 0 for i in range(12)}

    rank_changes      = []
    rank_history      = {i: [(0, r_init)] for i in range(12)}
    lambda_history    = {i: [] for i in range(12)}
    global_lmax_hist  = []
    running_loss      = 0.0
    steps_logged      = 0
    step              = 0
    run_start         = time.time()

    model.train()

    for epoch in range(epochs):
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            step        += 1
            steps_logged += 1

            optimizer.zero_grad()
            out  = model(pixel_values=imgs)
            loss = criterion(out.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

            # ── CGRS rank check every K steps ─────────────────────────────
            if step % K == 0:
                lmax_dict   = compute_perlayer_fisher(model, train_loader, N_PROBE)
                g_lmax      = global_lmax(lmax_dict)
                global_lmax_hist.append((step, g_lmax))

                for i in range(12):
                    for proj in ["query", "value"]:
                        lambda_history[i].append((step, lmax_dict.get((i, proj), 0.0)))

                changed_this_check = False
                for i in range(12):
                    if cooldown_left[i] > 0:
                        cooldown_left[i] = max(0, cooldown_left[i] - K)
                        continue

                    cur_r    = layer_ranks[i]
                    q_lmax   = lmax_dict.get((i, "query"), 0.0)
                    v_lmax   = lmax_dict.get((i, "value"), 0.0)
                    lay_lmax = max(q_lmax, v_lmax)

                    if is_per_layer_tau:
                        tau_i = max(
                            tau_config.get((i, "query"), pct_50),
                            tau_config.get((i, "value"), pct_50)
                        )
                    else:
                        tau_i = tau_config

                    should_expand = lay_lmax > tau_i and cur_r < R_MAX

                    if should_expand:
                        idx     = RANK_LIST.index(cur_r) if cur_r in RANK_LIST \
                                  else min(range(len(RANK_LIST)), key=lambda x: abs(RANK_LIST[x] - cur_r))
                        new_r   = RANK_LIST[min(idx + 1, len(RANK_LIST) - 1)]
                        if new_r != cur_r:
                            # Rebuild model at new global rank, restore all weights
                            cur_lr   = float(scheduler.get_last_lr()[0])
                            model    = transition_lora_rank(model, cur_r, new_r)
                            layer_ranks[i]   = new_r
                            cooldown_left[i] = COOLDOWN
                            changed_this_check = True
                            rank_history[i].append((step, new_r))
                            rank_changes.append({
                                "step": step, "layer": i,
                                "old_r": cur_r, "new_r": new_r,
                                "lambda": lay_lmax, "tau": tau_i,
                            })
                            remaining = max(total_steps - step, 1)
                            optimizer = get_optimizer(model, lr=cur_lr,
                                                      weight_decay=P4["weight_decay"])
                            scheduler = get_scheduler(optimizer, remaining)
                            print(f"  Step {step:5d} | Layer {i:2d} | "
                                  f"↑ {cur_r} → {new_r}  λ={lay_lmax:.5f} τ={tau_i:.5f}")

                if step % (K * 10) == 0:
                    val_acc, _ = evaluate(model, val_loader)
                    avg_r      = np.mean(list(layer_ranks.values()))
                    elapsed    = (time.time() - run_start) / 60
                    print(f"  Ep {epoch+1} | Step {step:5d} | "
                          f"Loss {running_loss/steps_logged:.4f} | "
                          f"Val {val_acc:.2f}% | AvgRank {avg_r:.1f} | "
                          f"λ_global {g_lmax:.5f} | {elapsed:.1f}m")
                    running_loss  = 0.0
                    steps_logged  = 0

        val_acc, _ = evaluate(model, val_loader)
        avg_r      = np.mean(list(layer_ranks.values()))
        print(f"\n  ── Epoch {epoch+1}/{epochs} done | "
              f"Val {val_acc:.2f}% | AvgRank {avg_r:.1f} | "
              f"Rank changes so far: {len(rank_changes)}")

    test_acc, test_loss = evaluate(model, test_loader)
    final_ranks         = dict(layer_ranks)
    avg_rank            = float(np.mean(list(final_ranks.values())))
    elapsed_total       = (time.time() - run_start) / 60

    print(f"\n✓ {run_name} COMPLETE")
    print(f"  Test Acc:    {test_acc:.2f}%  |  Test Loss: {test_loss:.4f}")
    print(f"  Avg Rank:    {avg_rank:.2f}   |  Final Ranks: {list(final_ranks.values())}")
    print(f"  LoRA Params: {count_trainable(model):,}  |  Rank Changes: {len(rank_changes)}")
    print(f"  Time:        {elapsed_total:.1f} min")

    result = {
        "run_name":          run_name,
        "tau":               "per-layer" if is_per_layer_tau else float(tau_config),
        "r_init":            r_init,
        "expand_only":       expand_only,
        "test_acc":          test_acc,
        "test_loss":         test_loss,
        "avg_rank":          avg_rank,
        "final_ranks":       {str(k): v for k, v in final_ranks.items()},
        "total_lora_params": count_trainable(model),
        "n_rank_changes":    len(rank_changes),
        "rank_changes":      rank_changes,
        "rank_history":      {str(k): v for k, v in rank_history.items()},
        "lambda_history":    {str(k): v for k, v in lambda_history.items()},
        "global_lmax_hist":  global_lmax_hist,
        "elapsed_min":       elapsed_total,
    }

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        rp = os.path.join(save_dir, f"{run_name}_result.json")
        with open(rp, "w") as f:
            json.dump(result, f, indent=2)
        cp = os.path.join(save_dir, f"{run_name}_checkpoint.pt")
        torch.save({"model_state_dict": model.state_dict(),
                    "final_ranks":      final_ranks}, cp)
        print(f"  Saved:      {rp}")

    del model
    torch.cuda.empty_cache()
    return result


print("Per-layer CGRS training loop ready.")
print("  Uses transition_lora_rank() from Phase 3 — SVD-based weight transfer.")
print("  r_init=16 matches Phase 3 starting point for fair comparison.")

Per-layer CGRS training loop ready.
  Uses transition_lora_rank() from Phase 3 — SVD-based weight transfer.
  r_init=16 matches Phase 3 starting point for fair comparison.


## Cell 8.5 · Corrected τ Calibration (Based on Live Training λ_max)

> **Why corrected:** All original Runs 1–3 collapsed to r=3 because Phase 2 τ values (~0.001–0.01) were derived from post-convergence checkpoints. Live training λ_max is ~100× smaller (range 0.00001–0.0001 observed in logs). These corrected τ values target the actual live signal and should produce real per-layer rank diversity.

In [9]:
# ── Cell 8.5: τ calibration for PEFT-native Fisher ──────────────────────────
# Phase 2 λ_max values at r=16 are the LIVE observed range (already loaded).
# These values represent the actual Fisher magnitudes when rank=16 is trained.
# r_init = 16 to be consistent with Phase 3 and give the model a fair start.

P4["r_init"] = 16   # ← CHANGED BACK to 16 (was 3, which was wrong for expand logic)

if PHASE2_LAYER_LMAX_R16:
    vals_r16 = np.array(list(PHASE2_LAYER_LMAX_R16.values()))
    pct_25   = float(np.percentile(vals_r16, 25))
    pct_50   = float(np.percentile(vals_r16, 50))
    pct_75   = float(np.percentile(vals_r16, 75))

    # Run C1: τ = 75th percentile → only sharp layers expand
    # Run C2: τ = 50th percentile → moderate sensitivity
    # Run C3: per-layer τ = each layer's own r=16 λ_max (the novelty)
    TAU_C1 = pct_75
    TAU_C2 = pct_50
    TAU_PER_LAYER = dict(PHASE2_LAYER_LMAX_R16)   # exact Phase 2 values as thresholds
else:
    # Hardcoded fallback from Phase 2 observed r=16 values in your JSON
    TAU_C1 = 0.003431    # r=16 λ_max 75th pct
    TAU_C2 = 0.001278    # r=16 λ_max 50th pct
    TAU_PER_LAYER = {
        (i, proj): v for (i, proj), v in [
            ((0,"query"), 5.44e-5), ((0,"value"), 3.58e-3),
            ((1,"query"), 3.38e-5), ((1,"value"), 5.94e-4),
            ((2,"query"), 1.15e-4), ((2,"value"), 1.28e-3),
            ((3,"query"), 3.55e-5), ((3,"value"), 3.63e-4),
            ((4,"query"), 4.03e-4), ((4,"value"), 3.88e-4),
            ((5,"query"), 1.54e-4), ((5,"value"), 4.42e-4),
            ((6,"query"), 8.06e-5), ((6,"value"), 5.31e-4),
            ((7,"query"), 3.34e-4), ((7,"value"), 2.18e-3),
            ((8,"query"), 6.39e-4), ((8,"value"), 1.43e-3),
            ((9,"query"), 2.54e-3), ((9,"value"), 2.10e-3),
            ((10,"query"),3.43e-3), ((10,"value"),1.15e-3),
            ((11,"query"),1.23e-2), ((11,"value"),2.71e-4),
        ]
    }

print(f"r_init = {P4['r_init']} (consistent with Phase 3)")
print(f"\nτ values (based on Phase 2 r=16 λ_max):")
print(f"  C1 (75th pct):  {TAU_C1:.6f}  — only layers sharper than most will expand")
print(f"  C2 (50th pct):  {TAU_C2:.6f}  — moderate, half the layers can expand")
print(f"  C3 (per-layer): each layer uses its own r=16 λ_max as its τ")
print(f"\nLogic: Layer expands rank when its live λ_max > τ for that layer.")
print(f"This means sharp layers (deep layers, especially L9-L11 query) will")
print(f"expand sooner, and flat layers (L0-L3 query) stay at r_init longer.")

r_init = 16 (consistent with Phase 3)

τ values (based on Phase 2 r=16 λ_max):
  C1 (75th pct):  0.001595  — only layers sharper than most will expand
  C2 (50th pct):  0.000486  — moderate, half the layers can expand
  C3 (per-layer): each layer uses its own r=16 λ_max as its τ

Logic: Layer expands rank when its live λ_max > τ for that layer.
This means sharp layers (deep layers, especially L9-L11 query) will
expand sooner, and flat layers (L0-L3 query) stay at r_init longer.


## Cell 9.5 — Run C1 (First Corrected Run)

**3 shared-τ runs + 1 per-layer-τ run (optional, highest novelty)**

Expected runtime per run: ~25–35 minutes on T4 GPU.  
Total: ~1.5–2.5 hours.  
Results are saved to Drive after each run — safe to interrupt and resume.

> **Run order recommendation:** Run 1 → Run 2 → Run 3 → Run 4 (optional)


In [10]:
# ── Cell 9.5: Delete old C1 result and re-run with corrected code ────────────
import os

# Delete stale result from the broken PerLayerLoRAViT run
old_c1 = os.path.join(PHASE4_DIR, "PL_Expand_C1_tau3e5_result.json")
old_c1_ckpt = os.path.join(PHASE4_DIR, "PL_Expand_C1_tau3e5_checkpoint.pt")
for f in [old_c1, old_c1_ckpt]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted stale file: {f}")

run_name = "PL_C1_tau75pct"
result   = load_if_exists(run_name, PHASE4_DIR)
if result is None:
    result = run_perlayer_cgrs(
        run_name,
        tau_config  = TAU_C1,
        r_init      = P4["r_init"],
        epochs      = P4["epochs"],
        save_dir    = PHASE4_DIR,
        expand_only = True,
    )
phase4_results[run_name] = result

✓ PL_C1_tau75pct already saved — Test Acc: 90.61%, Avg Rank: 39.17 — skipping.


## Cell 9.6 — Run C2 and C3 (only if C1 showed diversity)

> **Run these cells.** C1–C3 are corrected shared-τ runs. C4 is the corrected per-layer τ run. All use checkpoint/resume — if Colab disconnects, re-run and completed runs are skipped. Expected per-run time: ~25–35 min on T4.

In [11]:
# ── Cell 9.6: Run C2 and C3 (no skip condition — run all 3 always) ───────────
for run_name, tau in [("PL_C2_tau50pct", TAU_C2),
                      ("PL_C3_PerLayer",  TAU_PER_LAYER)]:
    result = load_if_exists(run_name, PHASE4_DIR)
    if result is None:
        result = run_perlayer_cgrs(
            run_name,
            tau_config  = tau,
            r_init      = P4["r_init"],
            epochs      = P4["epochs"],
            save_dir    = PHASE4_DIR,
            expand_only = True,
        )
    phase4_results[run_name] = result

combined_path = os.path.join(PHASE4_DIR, "phase4_results_combined.json")
with open(combined_path, "w") as f:
    json.dump(phase4_results, f, indent=2)

print(f"\n✓ All results saved: {combined_path}")
print(f"\n{'Run':<35} {'τ':>12} {'r_init':>6} {'AvgR':>6} {'TestAcc':>9} {'Changes':>8}")
print("-" * 80)
for rn, res in phase4_results.items():
    tau_str = "per-layer" if isinstance(res.get("tau"), str) else f"{res.get('tau', 0):.2e}"
    print(f"{rn:<35} {tau_str:>12} {res.get('r_init',16):>6} "
          f"{res.get('avg_rank',0):>6.1f} {res.get('test_acc',0):>8.2f}% "
          f"{res.get('n_rank_changes',0):>8}")

✓ PL_C2_tau50pct already saved — Test Acc: 90.72%, Avg Rank: 53.17 — skipping.
✓ PL_C3_PerLayer already saved — Test Acc: 90.85%, Avg Rank: 51.17 — skipping.

✓ All results saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/phase4_results_combined.json

Run                                            τ r_init   AvgR   TestAcc  Changes
--------------------------------------------------------------------------------
PL_C1_tau75pct                          1.59e-03     16   39.2    90.61%       17
PL_C2_tau50pct                          4.86e-04     16   53.2    90.72%       28
PL_C3_PerLayer                         per-layer     16   51.2    90.85%       26


## Cell 9.7 — Calibrate C3_Half τ Values
Cell 9.7 — C3_Half Calibration: τ = 0.5 * each layer's own Phase 2 lambda_max
This corrects the C3_PerLayer anomaly (negative curvature-rank correlation)
by halving each layer's threshold, making it easier for high-curvature layers
to exceed their tau and earn a rank expansion.

In [12]:
# Cell 9.7 — C3_Half Calibration (self-contained, safe to run standalone)

import os, json
import numpy as np

# ── Step 1: Rebuild PHASE2LAYERLMAXR16 if kernel was restarted ──────────────
if 'PHASE2LAYERLMAXR16' not in dir() or not PHASE2LAYERLMAXR16:
    print("PHASE2LAYERLMAXR16 not in memory — rebuilding from disk...")
    ALLRESULTSPATH = os.path.join(PHASE2_DIR, "all_results_complete.json")
    PHASE2LAYERLMAXR16 = {}

    if os.path.exists(ALLRESULTSPATH):
        with open(ALLRESULTSPATH) as f:
            all_results_p2 = json.load(f)

        curvature_data = all_results_p2.get("curvature", {})
        r16_block = curvature_data.get("r16", {})
        r16_per_layer = r16_block.get("per_layer", {})

        if r16_per_layer:
            for key_str, stats in r16_per_layer.items():
                try:
                    key = eval(key_str)
                    PHASE2LAYERLMAXR16[key] = stats.get("lambda_max", 0.0)
                except Exception:
                    pass
            vals = list(PHASE2LAYERLMAXR16.values())
            print(f"Loaded {len(PHASE2LAYERLMAXR16)} layer-proj pairs from disk.")
            print(f"  lambda_max range: {min(vals):.6e} → {max(vals):.6e}")
        else:
            print("WARNING: per_layer block missing in JSON — using hardcoded fallback.")
    else:
        print(f"WARNING: {ALLRESULTSPATH} not found — using hardcoded fallback.")

# ── Step 2: Hardcoded fallback if disk load also failed ─────────────────────
if not PHASE2LAYERLMAXR16:
    PHASE2LAYERLMAXR16 = {
        (0,  'query'): 5.44e-5,  (0,  'value'): 3.58e-3,
        (1,  'query'): 3.38e-5,  (1,  'value'): 5.94e-4,
        (2,  'query'): 1.15e-4,  (2,  'value'): 1.28e-3,
        (3,  'query'): 3.55e-5,  (3,  'value'): 3.63e-4,
        (4,  'query'): 4.03e-4,  (4,  'value'): 3.88e-4,
        (5,  'query'): 1.54e-4,  (5,  'value'): 4.42e-4,
        (6,  'query'): 8.06e-5,  (6,  'value'): 5.31e-4,
        (7,  'query'): 3.34e-4,  (7,  'value'): 2.18e-3,
        (8,  'query'): 6.39e-4,  (8,  'value'): 1.43e-3,
        (9,  'query'): 2.54e-3,  (9,  'value'): 2.10e-3,
        (10, 'query'): 3.43e-3,  (10, 'value'): 1.15e-3,
        (11, 'query'): 1.23e-2,  (11, 'value'): 2.71e-4,
    }
    print("Using hardcoded Phase 2 lambda_max values (from your research report).")

# ── Step 3: Build TAU_C3_HALF = 0.5 × each layer's own lambda_max ───────────
TAU_C3_HALF = {key: 0.5 * val for key, val in PHASE2LAYERLMAXR16.items()}

print(f"\nC3_Half tau values (tau_i = 0.5 × Phase 2 lambda_max):")
print(f"{'Layer-Proj':<20}  {'Phase2 lambda_max':>20}  {'C3_Half tau (×0.5)':>20}")
print("-" * 65)
for key in sorted(TAU_C3_HALF.keys()):
    orig = PHASE2LAYERLMAXR16[key]
    half = TAU_C3_HALF[key]
    print(f"{str(key):<20}  {orig:>20.6e}  {half:>20.6e}")

PHASE2LAYERLMAXR16 not in memory — rebuilding from disk...
Using hardcoded Phase 2 lambda_max values (from your research report).

C3_Half tau values (tau_i = 0.5 × Phase 2 lambda_max):
Layer-Proj               Phase2 lambda_max    C3_Half tau (×0.5)
-----------------------------------------------------------------
(0, 'query')                  5.440000e-05          2.720000e-05
(0, 'value')                  3.580000e-03          1.790000e-03
(1, 'query')                  3.380000e-05          1.690000e-05
(1, 'value')                  5.940000e-04          2.970000e-04
(2, 'query')                  1.150000e-04          5.750000e-05
(2, 'value')                  1.280000e-03          6.400000e-04
(3, 'query')                  3.550000e-05          1.775000e-05
(3, 'value')                  3.630000e-04          1.815000e-04
(4, 'query')                  4.030000e-04          2.015000e-04
(4, 'value')                  3.880000e-04          1.940000e-04
(5, 'query')                  1.5

## Cell 10 — Final Results & Plots

> Verifies all corrected runs completed. The original per-layer run is now handled in Cell 9.5 (Run C4).

In [13]:
# ── Cell 10: Final Ranks Per Layer — Improved Publication Plot ────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# ── Load all Phase 4 results from disk ───────────────────────────────────────
all_phase4 = {}
for fname in os.listdir(PHASE4_DIR):
    if fname.endswith("_result.json"):
        with open(os.path.join(PHASE4_DIR, fname)) as f:
            r = json.load(f)
        all_phase4[r["run_name"]] = r

# ── Phase 3 Global CGRS data (old logic) — hardcoded from report ─────────────
# These are GLOBAL runs — all 12 layers have the SAME final rank
PHASE3_RUNS = [
    {
        "run_name":    "CGRS_Global_tau0047",
        "tau":          0.0047,
        "avg_rank":     59.40,
        "test_acc":     90.02,
        "final_ranks":  {str(i): 64 for i in range(12)},  # global model — single rank
        "logic":        "Global CGRS (Phase 3 — Old Logic)",
        "tau_label":    "τ = 4.70e-03 (global shared)",
    },
    {
        "run_name":    "CGRS_Global_tau0200",
        "tau":          0.0200,
        "avg_rank":     36.08,
        "test_acc":     89.97,
        "final_ranks":  {str(i): 36 for i in range(12)},
        "logic":        "Global CGRS (Phase 3 — Old Logic)",
        "tau_label":    "τ = 2.00e-02 (global shared)",
    },
    {
        "run_name":    "CGRS_Global_tau0518",
        "tau":          0.0518,
        "avg_rank":     10.43,
        "test_acc":     87.96,
        "final_ranks":  {str(i): 10 for i in range(12)},
        "logic":        "Global CGRS (Phase 3 — Old Logic)",
        "tau_label":    "τ = 5.18e-02 (global shared)",
    },
]

# ── Phase 4 per-layer runs (new logic) ordered C1 → C2 → C3 ─────────────────
RUN_ORDER = ["PL_C1_tau75pct", "PL_C2_tau50pct", "PL_C3_PerLayer"]
TAU_LABELS = {
    "PL_C1_tau75pct":  "τ = 1.59e-03 (75th pct shared)",
    "PL_C2_tau50pct":  "τ = 4.86e-04 (50th pct shared)",
    "PL_C3_PerLayer":  "τ = per-layer (each layer's own r=16 λ_max)",
}
LOGIC_LABELS = {
    "PL_C1_tau75pct":  "Per-Layer CGRS (Phase 4 — New Logic)",
    "PL_C2_tau50pct":  "Per-Layer CGRS (Phase 4 — New Logic)",
    "PL_C3_PerLayer":  "Per-Layer CGRS (Phase 4 — New Logic) ★ Highest Novelty",
}

# ── Build ordered list: 3 old-logic + 3 new-logic ────────────────────────────
ordered_runs = []
for p3 in PHASE3_RUNS:
    ordered_runs.append(p3)
for rn in RUN_ORDER:
    if rn in all_phase4:
        res = all_phase4[rn]
        ordered_runs.append({
            "run_name":    rn,
            "tau":          res["tau"],
            "avg_rank":     res["avg_rank"],
            "test_acc":     res["test_acc"],
            "final_ranks":  res["final_ranks"],
            "logic":        LOGIC_LABELS[rn],
            "tau_label":    TAU_LABELS[rn],
        })

N = len(ordered_runs)
colors_old = "#7bafd4"   # muted blue for Phase 3
colors_new = "#2166ac"   # strong blue for Phase 4
star_color = "#d62728"   # red for the novelty C3 run

fig, axes = plt.subplots(1, N, figsize=(4.5 * N, 5), squeeze=False)

for idx, (ax, run) in enumerate(zip(axes[0], ordered_runs)):
    ranks = [run["final_ranks"].get(str(i), P4["r_init"]) for i in range(12)]
    is_new = "Phase 4" in run["logic"]
    is_star = "★" in run["logic"]

    bar_color = (star_color if is_star else (colors_new if is_new else colors_old))
    edgecolor = "black"

    bars = ax.bar(range(12), ranks, color=bar_color, edgecolor=edgecolor,
                  linewidth=0.8, alpha=0.88)

    # Add rank value labels on top of each bar
    for bar, rank in zip(bars, ranks):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                str(rank), ha="center", va="bottom", fontsize=7, fontweight="bold")

    ax.set_xlabel("Transformer Layer", fontsize=10)
    ax.set_ylabel("Final LoRA Rank", fontsize=10)
    ax.set_xticks(range(12))
    ax.set_ylim(0, P4["r_max"] + 10)

    # Reference lines
    ax.axhline(P4["r_min"], color="red",   linestyle="--", linewidth=1,
               label=f"r_min={P4['r_min']}")
    ax.axhline(P4["r_max"], color="green", linestyle="--", linewidth=1,
               label=f"r_max={P4['r_max']}")
    ax.legend(fontsize=7, loc="upper left")

    # Phase separator shading for old vs new
    ax.set_facecolor("#f7f7f7" if not is_new else "#eaf3fb")

    # Multi-line informative title
    phase_tag  = "── OLD LOGIC ──" if not is_new else "── NEW LOGIC ──"
    title_line1 = f"{phase_tag}"
    title_line2 = f"{run['logic']}"
    title_line3 = f"{run['tau_label']}"
    title_line4 = f"Acc={run['test_acc']:.2f}%  AvgRank={run['avg_rank']:.1f}"
    ax.set_title(
        f"{title_line1}\n{title_line2}\n{title_line3}\n{title_line4}",
        fontsize=8, fontweight="bold", pad=6,
        color=("darkred" if is_star else "navy" if is_new else "#333333")
    )

# Add vertical divider line between old and new logic panels
# Draw a line between subplot 3 and 4 using figure coordinates
fig.text(0.5, 0.5, "", transform=fig.transFigure)
for spine_ax in axes[0]:
    spine_ax.spines["top"].set_visible(False)
    spine_ax.spines["right"].set_visible(False)

# Super-title
fig.suptitle(
    "CGRS Final LoRA Rank per Transformer Layer\n"
    "Left 3: Global CGRS (Phase 3 — Old Logic) | Right 3: Per-Layer CGRS (Phase 4 — New Logic)",
    fontsize=12, fontweight="bold", y=1.02
)

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, "phase4_final_ranks_per_layer.png")
plt.savefig(plot_path, dpi=180, bbox_inches="tight")
plt.close()
print(f"\n✓ Improved plot saved: {plot_path}")


✓ Improved plot saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/plots/phase4_final_ranks_per_layer.png


## Cell 11 · Full Comparison Table (Phase 1–4)

In [14]:
# Build complete comparison table
print("\n── Complete Comparison: Phase 1 Fixed → Phase 3 Global CGRS → Phase 4 Per-Layer CGRS ──")
print(f"{'Method':<40} {'Type':<10} {'Avg Rank':>9} {'Test Acc':>9} {'LoRA Params':>12}")
print("=" * 82)

# Fixed baselines from Phase 1
fixed_baselines = [
    ("Fixed r=6",   "Static",  6,   87.27, 221184),
    ("Fixed r=10",  "Static", 10,   89.02, 368640),
    ("Fixed r=16",  "Static", 16,   89.51, 589824),
    ("Fixed r=64",  "Static", 64,   90.48, 2359296),
    ("Full FT",     "Static",  "–", 92.87, 86567396),
]
for name, t, r, acc, params in fixed_baselines:
    r_str = f"{r:>9}" if isinstance(r, int) else f"{'–':>9}"
    print(f"{name:<40} {t:<10} {r_str} {acc:>8.2f}% {params:>12,}")

print("-" * 82)

# Phase 3 Global CGRS (best runs)
p3_best = [
    ("CGRS Global τ=0.0047 (Ph3)",  "Dynamic", 59.40, 90.02, 2188000),
    ("CGRS Global τ=0.0200 (Ph3)",  "Dynamic", 36.08, 89.97, 1329000),
    ("CGRS Global τ=0.0518 (Ph3)",  "Dynamic", 10.43, 87.96,  384000),
]
for name, t, r, acc, params in p3_best:
    print(f"{name:<40} {t:<10} {r:>9.2f} {acc:>8.2f}% {params:>12,}")

print("-" * 82)

# Phase 4 Per-Layer CGRS
for run_name, res in phase4_results.items():
    label = run_name.replace("PL_CGRS_", "").replace("_", " ")
    t = "Per-Layer"
    r = res['avg_rank']
    acc = res['test_acc']
    params = res['total_lora_params']
    print(f"{'CGRS Per-Layer ' + label:<40} {t:<10} {r:>9.2f} {acc:>8.2f}% {params:>12,}")

print("=" * 82)



── Complete Comparison: Phase 1 Fixed → Phase 3 Global CGRS → Phase 4 Per-Layer CGRS ──
Method                                   Type        Avg Rank  Test Acc  LoRA Params
Fixed r=6                                Static             6    87.27%      221,184
Fixed r=10                               Static            10    89.02%      368,640
Fixed r=16                               Static            16    89.51%      589,824
Fixed r=64                               Static            64    90.48%    2,359,296
Full FT                                  Static             –    92.87%   86,567,396
----------------------------------------------------------------------------------
CGRS Global τ=0.0047 (Ph3)               Dynamic        59.40    90.02%    2,188,000
CGRS Global τ=0.0200 (Ph3)               Dynamic        36.08    89.97%    1,329,000
CGRS Global τ=0.0518 (Ph3)               Dynamic        10.43    87.96%      384,000
---------------------------------------------------------------

## Cell 12 · Publication-Quality Plots

In [15]:
import warnings
warnings.filterwarnings('ignore')

PLOT_DIR = os.path.join(PHASE4_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
    'axes.labelsize': 12, 'axes.titlesize': 13,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 9, 'legend.frameon': True,
    'legend.edgecolor': '#cccccc',
})

COLORS = {
    'fixed':    '#8e9aaf',
    'global':   '#457b9d',
    'perlayer': '#e63946',
    'fullft':   '#2d6a4f',
    'frozen':   '#b5b5b5',
}
LAYER_COLORS = plt.cm.plasma(np.linspace(0.1, 0.9, 12))


# ── Plot 1: Per-layer rank heatmap over training steps ────────────────────────
# Use the best per-layer run (per-layer tau or run with most rank changes)
def plot_rank_heatmap(result, title, save_path):
    rank_hist = result.get("rank_history", {})
    total_steps = P4["epochs"] * len(train_loader)

    # Build rank matrix [12 layers × N check steps]
    check_steps = sorted(set(
        s for layer_h in rank_hist.values() for s, r in layer_h
    ))
    if not check_steps:
        print(f"  Skipping heatmap for {title} — no rank history data")
        return

    rank_matrix = np.zeros((12, len(check_steps)))
    for i in range(12):
        layer_key = str(i)
        history = rank_hist.get(layer_key, [(0, P4["r_init"])])
        cur_rank = P4["r_init"]
        h_dict = {s: r for s, r in history}
        for j, step in enumerate(check_steps):
            if step in h_dict:
                cur_rank = h_dict[step]
            rank_matrix[i, j] = cur_rank

    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(rank_matrix, aspect='auto', cmap='YlOrRd',
                   vmin=P4["r_min"], vmax=P4["r_max"],
                   extent=[check_steps[0], check_steps[-1], 11.5, -0.5])
    cbar = plt.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label('LoRA Rank', fontsize=11)
    ax.set_xlabel('Training Step', fontsize=12)
    ax.set_ylabel('Transformer Layer', fontsize=12)
    ax.set_yticks(range(12))
    ax.set_yticklabels([f'L{i}' for i in range(12)], fontsize=9)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)

    # Add epoch boundaries
    steps_per_epoch = len(train_loader)
    for ep in range(1, P4["epochs"]):
        ax.axvline(ep * steps_per_epoch, color='white', linewidth=1.5,
                   linestyle='--', alpha=0.8, label=f'Epoch {ep}')

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


# ── Plot 2: Avg rank vs test accuracy frontier ────────────────────────────────
def plot_accuracy_frontier(phase4_results, save_path):
    fig, ax = plt.subplots(figsize=(9, 6))

    # Fixed baselines
    fixed_data = [(6, 87.27), (10, 89.02), (16, 89.51), (64, 90.48)]
    fx, fy = zip(*fixed_data)
    ax.plot(fx, fy, 's--', color=COLORS['fixed'], linewidth=1.5,
            markersize=8, label='Fixed-Rank LoRA (Phase 1)', zorder=2)

    # Phase 3 Global CGRS
    p3 = [(59.40, 90.02), (36.08, 89.97), (10.43, 87.96)]
    gx, gy = zip(*p3)
    ax.scatter(gx, gy, c=COLORS['global'], marker='D', s=90, zorder=3,
               label='Global CGRS (Phase 3)', edgecolors='white', linewidths=0.8)

    # Phase 4 Per-Layer CGRS
    pl_colors = ['#e63946', '#f4a261', '#2a9d8f', '#8338ec']
    for idx, (run_name, res) in enumerate(phase4_results.items()):
        label_short = run_name.replace("PL_CGRS_", "").replace("_", " ")
        ax.scatter(res['avg_rank'], res['test_acc'],
                   c=pl_colors[idx % len(pl_colors)], marker='o', s=130,
                   zorder=4, label=f'Per-Layer CGRS {label_short}',
                   edgecolors='black', linewidths=0.8)
        ax.annotate(f" {res['test_acc']:.2f}%", (res['avg_rank'], res['test_acc']),
                    fontsize=8.5, va='center')

    # References
    ax.axhline(92.87, color=COLORS['fullft'],  linestyle=':', linewidth=1.5, alpha=0.7,
               label='Full FT (92.87%)')
    ax.axhline(86.46, color=COLORS['frozen'],  linestyle=':', linewidth=1.5, alpha=0.7,
               label='Frozen backbone (86.46%)')

    ax.set_xlabel('Average LoRA Rank During Training', fontsize=12)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('Avg Rank vs Test Accuracy — Per-Layer CGRS vs Fixed LoRA vs Global CGRS\n'
                 '(ViT-Base/16, CIFAR-100, 3 epochs)', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8.5)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


# ── Plot 3: Per-layer final rank bar chart ────────────────────────────────────
def plot_final_ranks_by_layer(phase4_results, save_path):
    runs = list(phase4_results.values())
    n_runs = len(runs)
    x = np.arange(12)
    width = 0.8 / max(n_runs, 1)
    run_colors = ['#e63946', '#f4a261', '#2a9d8f', '#8338ec']

    fig, ax = plt.subplots(figsize=(13, 5))
    for idx, res in enumerate(runs):
        final_ranks = [res['final_ranks'].get(str(i), P4["r_init"]) for i in range(12)]
        label = res['run_name'].replace("PL_CGRS_", "").replace("_", " ")
        offset = (idx - n_runs / 2 + 0.5) * width
        bars = ax.bar(x + offset, final_ranks, width=width * 0.9,
                      color=run_colors[idx % len(run_colors)], alpha=0.85,
                      label=f'{label} (acc={res["test_acc"]:.2f}%)',
                      edgecolor='white', linewidth=0.5)

    ax.axhline(P4["r_init"], color='gray', linestyle='--', linewidth=1.2,
               alpha=0.6, label=f'r_init = {P4["r_init"]}')
    ax.set_xticks(x)
    ax.set_xticklabels([f'L{i}' for i in range(12)])
    ax.set_xlabel('Transformer Layer', fontsize=12)
    ax.set_ylabel('Final LoRA Rank', fontsize=12)
    ax.set_title('Final Per-Layer Rank Distribution — Phase 4 Per-Layer CGRS\n'
                 '(Deep layers expected to retain higher rank due to higher curvature)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=8.5)
    ax.set_ylim(0, P4["r_max"] + 8)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


# ── Plot 4: Global λ_max trajectory ──────────────────────────────────────────
def plot_lambda_trajectory(phase4_results, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=False)
    axes = axes.flatten()
    run_colors = ['#e63946', '#f4a261', '#2a9d8f', '#8338ec']

    for idx, (run_name, res) in enumerate(phase4_results.items()):
        ax = axes[idx]
        gl_hist = res.get("global_lmax_history", [])
        if gl_hist:
            steps, lmaxes = zip(*gl_hist)
            ax.semilogy(steps, lmaxes, color=run_colors[idx], linewidth=1.8)

        label = run_name.replace("PL_CGRS_", "").replace("_", " ")
        ax.set_title(f'{label}\nAcc={res["test_acc"]:.2f}% | AvgRank={res["avg_rank"]:.1f}',
                     fontsize=10, fontweight='bold')
        ax.set_xlabel('Training Step', fontsize=10)
        ax.set_ylabel('Global λ_max (log scale)', fontsize=9)
        ax.grid(True, alpha=0.3)
        steps_per_epoch = len(train_loader)
        for ep in range(1, P4["epochs"]):
            ax.axvline(ep * steps_per_epoch, color='gray', linewidth=1, linestyle='--', alpha=0.5)

    # Hide unused subplots
    for idx in range(len(phase4_results), 4):
        axes[idx].set_visible(False)

    plt.suptitle('Global λ_max Trajectory — Per-Layer CGRS Phase 4', 
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


# ── Generate all plots ────────────────────────────────────────────────────────
print("Generating Phase 4 plots...")

# Rank heatmap for each run
for run_name, res in phase4_results.items():
    heatmap_path = os.path.join(PLOT_DIR, f"{run_name}_rank_heatmap.png")
    plot_rank_heatmap(res, f"Per-Layer Rank Trajectory — {run_name.replace('PL_CGRS_','').replace('_',' ')}", 
                      heatmap_path)

# Frontier plot
frontier_path = os.path.join(PLOT_DIR, "phase4_accuracy_frontier.png")
plot_accuracy_frontier(phase4_results, frontier_path)

# Final rank bar chart
final_ranks_path = os.path.join(PLOT_DIR, "phase4_final_rank_distribution.png")
plot_final_ranks_by_layer(phase4_results, final_ranks_path)

# Lambda trajectory
lambda_path = os.path.join(PLOT_DIR, "phase4_lambda_trajectories.png")
plot_lambda_trajectory(phase4_results, lambda_path)

print("\n✓ All plots saved to:", PLOT_DIR)


Generating Phase 4 plots...
  Saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/plots/PL_C1_tau75pct_rank_heatmap.png
  Saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/plots/PL_C2_tau50pct_rank_heatmap.png
  Saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/plots/PL_C3_PerLayer_rank_heatmap.png
  Saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/plots/phase4_accuracy_frontier.png
  Saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/plots/phase4_final_rank_distribution.png
  Saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/plots/phase4_lambda_trajectories.png

✓ All plots saved to: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/plots


## Cell 13 · Per-Layer Curvature Allocation Analysis (Publication Table)

In [16]:
# Build a detailed per-layer analysis table for the paper
# Shows: Phase 2 λ_max at r=16 → Final rank in each Ph4 run → Supports hypothesis
print("\n── Per-Layer Allocation Analysis (Phase 4) ──────────────────────────────")
print(f"{'Layer':>6} {'Ph2 λ_Q':>12} {'Ph2 λ_V':>12}", end="")
for run_name in phase4_results:
    label = run_name.replace("PL_CGRS_","")[:12]
    print(f" {'r_'+label:>16}", end="")
print()
print("-" * (30 + 18 * len(phase4_results)))

for i in range(12):
    lq = PHASE2_LAYER_LMAX_R16.get((i, 'query'), 0.0)
    lv = PHASE2_LAYER_LMAX_R16.get((i, 'value'), 0.0)
    print(f"{'L'+str(i):>6} {lq:>12.5f} {lv:>12.5f}", end="")
    for run_name, res in phase4_results.items():
        final_r = res['final_ranks'].get(str(i), P4["r_init"])
        print(f" {final_r:>18}", end="")
    print()

print("\n── Key finding to validate for publication: ─────────────────────────────")
print("  Layers 9–11 (high λ_max Q) should end with higher rank than Layers 0–3")
print("  This directly validates the Phase 2 per-layer curvature hypothesis")

# Compute rank-curvature correlation per layer across runs
for run_name, res in phase4_results.items():
    final_ranks_list = [res['final_ranks'].get(str(i), P4["r_init"]) for i in range(12)]
    ph2_lmax_q = [PHASE2_LAYER_LMAX_R16.get((i, 'query'), 0.0) for i in range(12)]
    ph2_lmax_v = [PHASE2_LAYER_LMAX_R16.get((i, 'value'), 0.0) for i in range(12)]
    ph2_combined = [max(q, v) for q, v in zip(ph2_lmax_q, ph2_lmax_v)]

    if len(set(final_ranks_list)) > 1:
        r_pearson, p_val = stats.pearsonr(ph2_combined, final_ranks_list)
        r_spearman, _ = stats.spearmanr(ph2_combined, final_ranks_list)
        label = run_name.replace("PL_CGRS_", "")
        print(f"  {label}: Pearson r={r_pearson:.4f}, Spearman r={r_spearman:.4f} (p={p_val:.4f})")
    else:
        label = run_name.replace("PL_CGRS_", "")
        print(f"  {label}: All layers converged to same rank (r={final_ranks_list[0]})")



── Per-Layer Allocation Analysis (Phase 4) ──────────────────────────────
 Layer      Ph2 λ_Q      Ph2 λ_V   r_PL_C1_tau75p   r_PL_C2_tau50p   r_PL_C3_PerLay
------------------------------------------------------------------------------------
    L0      0.00005      0.00358                 52                 64                 52
    L1      0.00003      0.00059                 52                 64                 64
    L2      0.00012      0.00128                 64                 64                 64
    L3      0.00004      0.00036                 16                 64                 64
    L4      0.00040      0.00039                 16                 16                 30
    L5      0.00015      0.00044                 16                 16                 64
    L6      0.00008      0.00053                 16                 30                 64
    L7      0.00033      0.00218                 30                 64                 16
    L8      0.00064      0.00143    

## Cell 14 · Save Complete Cross-Phase Results JSON

In [17]:
# ── Cell 14: Save Complete Cross-Phase Results & Config ─────────────────────
import os, json

# Build phase4 summary dict from phase4_results (populated by Cells 9.5 + 9.6)
all_results_updated = {}

# Incorporate Phase 1 results
all_results_updated["phase1"] = {k: v for k, v in phase1_results.items()}

# Incorporate Phase 2 curvature data
all_results_updated["phase2"] = {
    "layer_lmax_r16": {str(k): v for k, v in PHASE2_LAYER_LMAX_R16.items()}
}

# Incorporate Phase 3 results (hardcoded from report — not in this notebook's memory)
all_results_updated["phase3"] = {
    "CGRS_tau0047":  {"tau": 0.0047,  "avg_rank": 59.40, "test_acc": 90.02, "lora_params": 2188000},
    "CGRS_tau0200":  {"tau": 0.0200,  "avg_rank": 36.08, "test_acc": 89.97, "lora_params": 1329000},
    "CGRS_tau0518":  {"tau": 0.0518,  "avg_rank": 10.43, "test_acc": 87.96, "lora_params": 384000},
}

# Incorporate Phase 4 results
all_results_updated["phase4"] = {
    run_name: {
        "test_acc":         res["test_acc"],
        "test_loss":        res["test_loss"],
        "avg_rank":         res["avg_rank"],
        "final_ranks":      res["final_ranks"],
        "total_lora_params": res["total_lora_params"],
        "n_rank_changes":   res["n_rank_changes"],
        "tau":              res["tau"],
        "elapsed_min":      res.get("elapsed_min", 0),
    }
    for run_name, res in phase4_results.items()
}

# Save updated complete results — use PHASE4_DIR, not DRIVE_BASE (not defined here)
updated_path = os.path.join(PHASE4_DIR, "all_results_complete_phase4.json")
with open(updated_path, "w") as f:
    json.dump(all_results_updated, f, indent=2)
print(f"✓ Saved complete cross-phase results to: {updated_path}")

# Save Phase 4 config — use TAU_C1/TAU_C2/TAU_PER_LAYER (TAU_RUNS is not defined)
config_p4 = {
    **P4,
    "tau_runs": {
        "C1_tau75pct": float(TAU_C1),
        "C2_tau50pct": float(TAU_C2),
    },
    "tau_per_layer": {str(k): float(v) for k, v in TAU_PER_LAYER.items()},
    "model_name":   "google/vit-base-patch16-224",
    "num_classes":  100,
    "random_seed":  SEED,   # SEED is defined in Cell 3, not RANDOM_SEED
}
config_path = os.path.join(PHASE4_DIR, "config_phase4.json")
with open(config_path, "w") as f:
    json.dump(config_p4, f, indent=2)
print(f"✓ Saved Phase 4 config to: {config_path}")

print("\n─────────────────────────────────────────────────────────────────────────")
print("Phase 4 COMPLETE.")
print("\nFiles saved:")
print(f"  {updated_path}")
print(f"  {config_path}")
print("─────────────────────────────────────────────────────────────────────────")

✓ Saved complete cross-phase results to: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/all_results_complete_phase4.json
✓ Saved Phase 4 config to: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/config_phase4.json

─────────────────────────────────────────────────────────────────────────
Phase 4 COMPLETE.

Files saved:
  /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/all_results_complete_phase4.json
  /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/config_phase4.json
─────────────────────────────────────────────────────────────────────────


# ── Cell 12A: Accuracy vs. LoRA Params (Pareto Frontier) ─────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

os.makedirs(os.path.join(PHASE4_DIR, "plots"), exist_ok=True)
PLOT_OUT = os.path.join(PHASE4_DIR, "plots")

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(7, 5))

# ── Phase 1: Fixed LoRA ───────────────────────────────────────────────────────
fixed_lora = [
    ("r=3",   84.77, 110_000),
    ("r=5",   87.36, 184_000),
    ("r=6",   87.27, 221_000),
    ("r=10",  89.02, 369_000),
    ("r=12",  89.31, 442_000),
    ("r=16",  89.51, 590_000),
    ("r=30",  90.30, 1_100_000),
    ("r=52",  90.62, 1_920_000),
    ("r=64",  90.48, 2_360_000),
    ("r=80",  90.42, 2_950_000),
    ("r=100", 90.90, 3_700_000),
    ("r=128", 90.55, 4_700_000),
    ("r=150", 90.70, 5_500_000),
    ("r=200", 90.95, 7_400_000),
]
fl_params = [x[2] / 1e6 for x in fixed_lora]
fl_acc    = [x[1] for x in fixed_lora]
ax.plot(fl_params, fl_acc, color="#555555", linewidth=1.2,
        linestyle="-", marker="o", markersize=4, zorder=2,
        label="Fixed LoRA (Phase 1)")

# ── Phase 3: Global CGRS ──────────────────────────────────────────────────────
global_cgrs = [
    (87.96, 384_000),
    (89.97, 1_329_000),
    (90.02, 2_188_000),
]
gc_params = [x[1] / 1e6 for x in global_cgrs]
gc_acc    = [x[0] for x in global_cgrs]
ax.scatter(gc_params, gc_acc, color="#d6604d", s=70, zorder=4,
           marker="s", label="Global CGRS (Phase 3)")

# ── Phase 4: Per-Layer CGRS ───────────────────────────────────────────────────
# Load from phase4_results dict (populated in Cells 9.5/9.6)
PHASE4_RUN_ORDER = ["PL_C1_tau75pct", "PL_C2_tau50pct", "PL_C3_PerLayer"]
PHASE4_LABELS    = {
    "PL_C1_tau75pct": r"Per-Layer CGRS, $\tau$=75th pct",
    "PL_C2_tau50pct": r"Per-Layer CGRS, $\tau$=50th pct",
    "PL_C3_PerLayer": r"Per-Layer CGRS, $\tau$=per-layer",
}
PHASE4_MARKERS   = {"PL_C1_tau75pct": "^", "PL_C2_tau50pct": "D", "PL_C3_PerLayer": "*"}
PHASE4_COLORS    = {"PL_C1_tau75pct": "#4393c3", "PL_C2_tau50pct": "#2166ac", "PL_C3_PerLayer": "#053061"}

for rn in PHASE4_RUN_ORDER:
    if rn in phase4_results:
        res = phase4_results[rn]
        p   = res["total_lora_params"] / 1e6
        acc = res["test_acc"]
        ax.scatter(p, acc,
                   color=PHASE4_COLORS[rn],
                   marker=PHASE4_MARKERS[rn],
                   s=120 if rn == "PL_C3_PerLayer" else 80,
                   zorder=5, label=PHASE4_LABELS[rn])

# ── Reference lines ───────────────────────────────────────────────────────────
ax.axhline(92.87, color="#888888", linestyle=":", linewidth=1.0)
ax.text(0.12, 92.95, "Full fine-tune (92.87%)",
        fontsize=8, color="#666666", va="bottom")

ax.axhline(86.46, color="#aaaaaa", linestyle=":", linewidth=0.8)
ax.text(0.12, 86.54, "Frozen backbone (86.46%)",
        fontsize=8, color="#888888", va="bottom")

ax.set_xlabel("LoRA Parameter Count (millions)", fontsize=11)
ax.set_ylabel("Test Accuracy (%)", fontsize=11)
ax.set_xlim(left=0)
ax.set_ylim(84, 93.5)
ax.legend(fontsize=8, loc="lower right", framealpha=0.9)
ax.tick_params(labelsize=9)

plt.tight_layout()
save_path = os.path.join(PLOT_OUT, "pareto_frontier.pdf")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.savefig(save_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {save_path}")

In [18]:
# ── Cell 12A: Accuracy vs. LoRA Params (Pareto Frontier) ─────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

os.makedirs(os.path.join(PHASE4_DIR, "plots"), exist_ok=True)
PLOT_OUT = os.path.join(PHASE4_DIR, "plots")

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(7, 5))

# ── Phase 1: Fixed LoRA ───────────────────────────────────────────────────────
fixed_lora = [
    ("r=3",   84.77, 110_000),
    ("r=5",   87.36, 184_000),
    ("r=6",   87.27, 221_000),
    ("r=10",  89.02, 369_000),
    ("r=12",  89.31, 442_000),
    ("r=16",  89.51, 590_000),
    ("r=30",  90.30, 1_100_000),
    ("r=52",  90.62, 1_920_000),
    ("r=64",  90.48, 2_360_000),
    ("r=80",  90.42, 2_950_000),
    ("r=100", 90.90, 3_700_000),
    ("r=128", 90.55, 4_700_000),
    ("r=150", 90.70, 5_500_000),
    ("r=200", 90.95, 7_400_000),
]
fl_params = [x[2] / 1e6 for x in fixed_lora]
fl_acc    = [x[1] for x in fixed_lora]
ax.plot(fl_params, fl_acc, color="#555555", linewidth=1.2,
        linestyle="-", marker="o", markersize=4, zorder=2,
        label="Fixed LoRA (Phase 1)")

# ── Phase 3: Global CGRS ──────────────────────────────────────────────────────
global_cgrs = [
    (87.96, 384_000),
    (89.97, 1_329_000),
    (90.02, 2_188_000),
]
gc_params = [x[1] / 1e6 for x in global_cgrs]
gc_acc    = [x[0] for x in global_cgrs]
ax.scatter(gc_params, gc_acc, color="#d6604d", s=70, zorder=4,
           marker="s", label="Global CGRS (Phase 3)")

# ── Phase 4: Per-Layer CGRS ───────────────────────────────────────────────────
# Load from phase4_results dict (populated in Cells 9.5/9.6)
PHASE4_RUN_ORDER = ["PL_C1_tau75pct", "PL_C2_tau50pct", "PL_C3_PerLayer"]
PHASE4_LABELS    = {
    "PL_C1_tau75pct": r"Per-Layer CGRS, $\tau$=75th pct",
    "PL_C2_tau50pct": r"Per-Layer CGRS, $\tau$=50th pct",
    "PL_C3_PerLayer": r"Per-Layer CGRS, $\tau$=per-layer",
}
PHASE4_MARKERS   = {"PL_C1_tau75pct": "^", "PL_C2_tau50pct": "D", "PL_C3_PerLayer": "*"}
PHASE4_COLORS    = {"PL_C1_tau75pct": "#4393c3", "PL_C2_tau50pct": "#2166ac", "PL_C3_PerLayer": "#053061"}

for rn in PHASE4_RUN_ORDER:
    if rn in phase4_results:
        res = phase4_results[rn]
        p   = res["total_lora_params"] / 1e6
        acc = res["test_acc"]
        ax.scatter(p, acc,
                   color=PHASE4_COLORS[rn],
                   marker=PHASE4_MARKERS[rn],
                   s=120 if rn == "PL_C3_PerLayer" else 80,
                   zorder=5, label=PHASE4_LABELS[rn])

# ── Reference lines ───────────────────────────────────────────────────────────
ax.axhline(92.87, color="#888888", linestyle=":", linewidth=1.0)
ax.text(0.12, 92.95, "Full fine-tune (92.87%)",
        fontsize=8, color="#666666", va="bottom")

ax.axhline(86.46, color="#aaaaaa", linestyle=":", linewidth=0.8)
ax.text(0.12, 86.54, "Frozen backbone (86.46%)",
        fontsize=8, color="#888888", va="bottom")

ax.set_xlabel("LoRA Parameter Count (millions)", fontsize=11)
ax.set_ylabel("Test Accuracy (%)", fontsize=11)
ax.set_xlim(left=0)
ax.set_ylim(84, 93.5)
ax.legend(fontsize=8, loc="lower right", framealpha=0.9)
ax.tick_params(labelsize=9)

plt.tight_layout()
save_path = os.path.join(PLOT_OUT, "pareto_frontier.pdf")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.savefig(save_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {save_path}")

✓ Saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/plots/pareto_frontier.pdf


In [19]:
# ── Cell 12B: Per-Layer λ_max Heatmap at r=16 ────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

plt.style.use("seaborn-v0_8-white")

# Build 2×12 matrix: rows = [query, value], cols = layers 0-11
lmax_matrix = np.zeros((2, 12))
proj_labels  = ["query", "value"]
for i in range(12):
    for pi, proj in enumerate(proj_labels):
        lmax_matrix[pi, i] = PHASE2_LAYER_LMAX_R16.get((i, proj), 0.0)

fig, ax = plt.subplots(figsize=(9, 2.6))

# Log scale colormap — the range is 200× so log normalisation is essential
norm = mcolors.LogNorm(vmin=lmax_matrix[lmax_matrix > 0].min(),
                       vmax=lmax_matrix.max())
im = ax.imshow(lmax_matrix, cmap="YlOrRd", norm=norm, aspect="auto")

# Annotate cells with scientific notation
for pi in range(2):
    for li in range(12):
        val = lmax_matrix[pi, li]
        text_color = "white" if norm(val) > 0.65 else "black"
        ax.text(li, pi, f"{val:.1e}", ha="center", va="center",
                fontsize=7, color=text_color, fontweight="medium")

ax.set_xticks(range(12))
ax.set_xticklabels([f"L{i}" for i in range(12)], fontsize=9)
ax.set_yticks([0, 1])
ax.set_yticklabels(["Query", "Value"], fontsize=9)

cbar = fig.colorbar(im, ax=ax, orientation="vertical", pad=0.02, fraction=0.03)
cbar.set_label(r"$\lambda_{\max}$ (diagonal Fisher, $r$=16)", fontsize=9)
cbar.ax.tick_params(labelsize=8)

ax.set_xlabel("Transformer Block Index", fontsize=10)

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
save_path = os.path.join(PLOT_OUT, "lmax_heatmap_r16.pdf")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.savefig(save_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {save_path}")

✓ Saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/plots/lmax_heatmap_r16.pdf


In [20]:
# ── Cell 12C: λ_max vs. Final Rank Correlation (Cell 13 logic, visual) ────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")

# Aggregate λ_max per layer: max over query and value
layer_lmax  = np.array([
    max(PHASE2_LAYER_LMAX_R16.get((i, "query"), 0.0),
        PHASE2_LAYER_LMAX_R16.get((i, "value"), 0.0))
    for i in range(12)
])

TARGET_RUNS = {
    "PL_C1_tau75pct": (r"$\tau$=75th pct", "#4393c3", "o"),
    "PL_C2_tau50pct": (r"$\tau$=50th pct", "#2166ac", "s"),
    "PL_C3_PerLayer": (r"$\tau$=per-layer", "#053061", "^"),
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)

for ax, (rn, (label, color, marker)) in zip(axes, TARGET_RUNS.items()):
    if rn not in phase4_results:
        ax.text(0.5, 0.5, f"{rn}\nnot found", ha="center", va="center",
                transform=ax.transAxes, fontsize=9, color="gray")
        continue

    fr = phase4_results[rn]["final_ranks"]
    final_ranks = np.array([int(fr[str(i)]) for i in range(12)])

    r_p, p_p = stats.pearsonr(layer_lmax, final_ranks)
    r_s, p_s = stats.spearmanr(layer_lmax, final_ranks)

    # Regression line
    slope, intercept, *_ = stats.linregress(layer_lmax, final_ranks)
    x_line = np.linspace(layer_lmax.min(), layer_lmax.max(), 100)
    ax.plot(x_line, slope * x_line + intercept,
            color="#999999", linewidth=1.0, linestyle="--", zorder=1)

    sc = ax.scatter(layer_lmax, final_ranks,
                    c=color, marker=marker, s=70, zorder=3, edgecolors="white",
                    linewidths=0.5)

    for i in range(12):
        ax.annotate(f"L{i}", (layer_lmax[i], final_ranks[i]),
                    textcoords="offset points", xytext=(5, 3),
                    fontsize=6.5, color="#444444")

    stat_text = (f"Pearson $r$ = {r_p:.3f} ($p$={p_p:.3f})\n"
                 f"Spearman $\\rho$ = {r_s:.3f} ($p$={p_s:.3f})")
    ax.text(0.04, 0.97, stat_text, transform=ax.transAxes,
            fontsize=7.5, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7, ec="none"))

    ax.set_xlabel(r"Phase 2 $\lambda_{\max}$ ($r$=16)", fontsize=9)
    ax.set_ylabel("Final LoRA Rank (Phase 4)", fontsize=9)
    ax.text(0.98, 0.02, label, transform=ax.transAxes,
            fontsize=8, va="bottom", ha="right", color=color, fontweight="bold")
    ax.tick_params(labelsize=8)

plt.tight_layout()
save_path = os.path.join(PLOT_OUT, "correlation_lmax_vs_rank.pdf")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.savefig(save_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {save_path}")

✓ Saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/plots/correlation_lmax_vs_rank.pdf


In [21]:
# ── Cell 12D: Rank Evolution Over Training Steps ─────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")

TARGET_RUN = "PL_C3_PerLayer"   # highest-novelty run; change to any run name

if TARGET_RUN not in phase4_results:
    print(f"⚠ {TARGET_RUN} not in phase4_results — skipping Cell 12D")
else:
    rh = phase4_results[TARGET_RUN]["rank_history"]

    fig, ax = plt.subplots(figsize=(9, 4.5))

    # Colour layers by index: early=light, deep=dark
    cmap   = cm.get_cmap("Blues", 14)
    r_init = phase4_results[TARGET_RUN].get("r_init", 16)

    # Compute total steps for x-axis
    all_steps = []
    for i in range(12):
        history = rh.get(str(i), [[0, r_init]])
        all_steps.extend([pt[0] for pt in history])
    max_step = max(all_steps) if all_steps else 2100

    for i in range(12):
        history = rh.get(str(i), [[0, r_init]])
        steps   = [0] + [pt[0] for pt in history] + [max_step]
        ranks   = [r_init] + [pt[1] for pt in history] + [history[-1][1]]

        # Step function: each rank persists until next change
        step_x, step_y = [steps[0]], [ranks[0]]
        for j in range(1, len(steps)):
            step_x.append(steps[j])
            step_y.append(ranks[j - 1])
            step_x.append(steps[j])
            step_y.append(ranks[j])

        linewidth = 1.5 if i >= 9 else 0.9
        alpha     = 1.0 if i >= 9 else 0.65
        color     = cmap(4 + i)
        ax.plot(step_x, step_y, linewidth=linewidth, alpha=alpha, color=color,
                label=f"L{i}")

    ax.set_xlabel("Training Step", fontsize=11)
    ax.set_ylabel("LoRA Rank", fontsize=11)
    ax.set_yticks([3, 5, 6, 10, 12, 16, 30, 52, 64])
    ax.set_xlim(0, max_step)
    ax.set_ylim(0, 70)

    # Epoch markers (3 epochs, 703 steps each for 45k/64 batch)
    steps_per_epoch = len(train_loader)
    for e in [1, 2]:
        ax.axvline(e * steps_per_epoch, color="#cccccc",
                   linewidth=0.8, linestyle=":", zorder=0)
        ax.text(e * steps_per_epoch + 20, 67, f"Epoch {e+1}",
                fontsize=7.5, color="#999999", va="top")

    # Legend: only deep layers labelled explicitly, rest grouped
    handles, labels = ax.get_legend_handles_labels()
    # Show all layers in two columns
    ax.legend(handles, labels, fontsize=7, ncol=4, loc="upper left",
              framealpha=0.85, handlelength=1.5)

    ax.text(0.99, 0.02,
            r"$\tau$ = per-layer ($\lambda_{\max}$ at $r$=16)",
            transform=ax.transAxes, fontsize=8, ha="right", va="bottom",
            color="#444444")

    plt.tight_layout()
    save_path = os.path.join(PLOT_OUT, "rank_evolution_C3.pdf")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.savefig(save_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {save_path}")

✓ Saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/plots/rank_evolution_C3.pdf


In [22]:
# ── Cell 12E: Global λ_max Trajectory Over Training Steps ────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")

TRAJ_RUNS = {
    "PL_C1_tau75pct": (r"$\tau$=75th pct (1.59e$-$03)",  "#4393c3", "-"),
    "PL_C2_tau50pct": (r"$\tau$=50th pct (4.86e$-$04)",  "#2166ac", "--"),
    "PL_C3_PerLayer": (r"$\tau$=per-layer",               "#053061", "-."),
}

fig, ax = plt.subplots(figsize=(8, 4))

for rn, (label, color, ls) in TRAJ_RUNS.items():
    if rn not in phase4_results:
        continue
    hist = phase4_results[rn].get("global_lmax_hist", [])
    if not hist:
        continue
    steps  = [pt[0] for pt in hist]
    lmax_v = [pt[1] for pt in hist]
    ax.plot(steps, lmax_v, color=color, linewidth=1.5,
            linestyle=ls, marker=".", markersize=5, label=label)

# τ reference lines for C1 and C2 (shared scalars)
ax.axhline(TAU_C1, color="#4393c3", linewidth=0.7, linestyle=":",
           label=rf"$\tau_{{C1}}$ = {TAU_C1:.2e}")
ax.axhline(TAU_C2, color="#2166ac", linewidth=0.7, linestyle=":",
           label=rf"$\tau_{{C2}}$ = {TAU_C2:.2e}")

# Epoch markers
steps_per_epoch = len(train_loader)
for e in [1, 2]:
    ax.axvline(e * steps_per_epoch, color="#cccccc",
               linewidth=0.8, linestyle=":", zorder=0)
    ax.text(e * steps_per_epoch + 20, ax.get_ylim()[1] * 0.97,
            f"Epoch {e+1}", fontsize=7.5, color="#999999", va="top")

ax.set_xlabel("Training Step", fontsize=11)
ax.set_ylabel(r"Global $\lambda_{\max}$ (diagonal Fisher)", fontsize=11)
ax.set_yscale("log")
ax.legend(fontsize=8, loc="upper right", framealpha=0.9)
ax.tick_params(labelsize=9)

plt.tight_layout()
save_path = os.path.join(PLOT_OUT, "lmax_trajectory_all_runs.pdf")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.savefig(save_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {save_path}")

✓ Saved: /home1/jkshaikh/CGRS_Project/CGRS_Buildup/phase4_results/plots/lmax_trajectory_all_runs.pdf


In [23]:
#This was the Phase 4 Per-Layer-CGRS Implementation

In [24]:
# ── Figure 1: Per-Layer Diagonal Fisher λ_max at r=16 (Phase 2) ──────────────
# Motivation figure: shows the 200× curvature range that justifies per-layer τ
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np, os

PLOT_DIR = os.path.join(PHASE4_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

lmax_matrix = np.zeros((2, 12))
for i in range(12):
    for pi, proj in enumerate(["query", "value"]):
        lmax_matrix[pi, i] = PHASE2_LAYER_LMAX_R16.get((i, proj), 0.0)

fig, ax = plt.subplots(figsize=(9, 2.4))

norm = mcolors.LogNorm(
    vmin=lmax_matrix[lmax_matrix > 0].min(),
    vmax=lmax_matrix.max()
)
im = ax.imshow(lmax_matrix, cmap="YlOrRd", norm=norm, aspect="auto")

for pi in range(2):
    for li in range(12):
        val = lmax_matrix[pi, li]
        tc = "white" if norm(val) > 0.62 else "black"
        ax.text(li, pi, f"{val:.1e}", ha="center", va="center",
                fontsize=7.5, color=tc)

ax.set_xticks(range(12))
ax.set_xticklabels([f"L{i}" for i in range(12)])
ax.set_yticks([0, 1])
ax.set_yticklabels(["Query", "Value"])
ax.set_xlabel("Transformer Block Index")

for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(im, ax=ax, orientation="vertical",
                    pad=0.015, fraction=0.025)
cbar.set_label(r"$\lambda_{\max}$ (diagonal Fisher, $r$=16)", fontsize=9)
cbar.ax.tick_params(labelsize=8)

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(os.path.join(PLOT_DIR, f"fig1_lmax_heatmap.{ext}"),
                bbox_inches="tight")
plt.show()
print("✓ Figure 1 saved.")

✓ Figure 1 saved.


In [25]:
# ── Figure 2: Accuracy vs. LoRA Parameter Count — Pareto Frontier ────────────
# Main result: Per-Layer CGRS vs Global CGRS vs Fixed-Rank LoRA vs Full FT
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, os

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "legend.fontsize": 9,
    "legend.frameon": True,
    "legend.edgecolor": "#cccccc",
    "legend.framealpha": 0.9,
})

fig, ax = plt.subplots(figsize=(7.5, 5.2))

# ── Fixed-rank LoRA (Phase 1) — line connecting points ───────────────────────
fixed = [
    (221_184,  87.27),
    (368_640,  89.02),
    (589_824,  89.51),
    (1_100_000, 90.30),
    (1_920_000, 90.62),
    (2_359_296, 90.48),
    (3_700_000, 90.90),
]
fx = [p / 1e6 for p, _ in fixed]
fy = [a for _, a in fixed]
ax.plot(fx, fy, color="#888888", linewidth=1.4, linestyle="-",
        marker="o", markersize=5, markerfacecolor="white",
        markeredgewidth=1.2, markeredgecolor="#888888",
        label="Fixed-rank LoRA", zorder=2)

# Annotate a few fixed-rank points
for params, acc, lbl in [(221_184, 87.27, "r=6"),
                          (589_824, 89.51, "r=16"),
                          (2_359_296, 90.48, "r=64")]:
    ax.annotate(lbl, (params / 1e6, acc),
                textcoords="offset points", xytext=(5, -10),
                fontsize=7.5, color="#666666")

# ── Global CGRS (Phase 3) ─────────────────────────────────────────────────────
p3_pts = [
    (384_000,   87.96, r"$\tau$=5.18e-02"),
    (1_329_000, 89.97, r"$\tau$=2.00e-02"),
    (2_188_000, 90.02, r"$\tau$=4.70e-03"),
]
p3x = [p / 1e6 for p, _, _ in p3_pts]
p3y = [a for _, a, _ in p3_pts]
ax.plot(p3x, p3y, color="#4393c3", linewidth=1.2, linestyle="--",
        marker="D", markersize=6, markerfacecolor="white",
        markeredgewidth=1.4, markeredgecolor="#4393c3",
        label="Global CGRS (Phase 3)", zorder=3)

# ── Per-Layer CGRS (Phase 4) ──────────────────────────────────────────────────
# Ordered by increasing params for a connected curve
pl_order = ["PL_C1_tau75pct", "PL_C2_tau50pct", "PL_C3_PerLayer"]
pl_labels = {
    "PL_C1_tau75pct": r"$\tau$=75th pct",
    "PL_C2_tau50pct": r"$\tau$=50th pct",
    "PL_C3_PerLayer": r"$\tau$=per-layer",
}
pl_pts = []
for rn in pl_order:
    if rn in phase4_results:
        res = phase4_results[rn]
        pl_pts.append((res["total_lora_params"] / 1e6,
                       res["test_acc"], pl_labels[rn]))

if pl_pts:
    pl_pts_sorted = sorted(pl_pts, key=lambda x: x[0])
    plx = [p for p, _, _ in pl_pts_sorted]
    ply = [a for _, a, _ in pl_pts_sorted]
    ax.plot(plx, ply, color="#d62728", linewidth=1.4, linestyle="-",
            marker="s", markersize=6, markerfacecolor="white",
            markeredgewidth=1.4, markeredgecolor="#d62728",
            label="Per-Layer CGRS (Phase 4)", zorder=4)
    for px, pa, plbl in pl_pts_sorted:
        ax.annotate(plbl, (px, pa),
                    textcoords="offset points", xytext=(5, 4),
                    fontsize=7.5, color="#d62728")

# ── Reference lines ───────────────────────────────────────────────────────────
ax.axhline(92.87, color="#2d6a4f", linewidth=0.9, linestyle=":",
           zorder=1)
ax.text(0.1, 93.05, "Full fine-tune  92.87%",
        fontsize=8, color="#2d6a4f", va="bottom")

ax.axhline(86.46, color="#aaaaaa", linewidth=0.9, linestyle=":",
           zorder=1)
ax.text(0.1, 86.62, "Frozen backbone  86.46%",
        fontsize=8, color="#aaaaaa", va="bottom")

ax.set_xlabel("LoRA Parameter Count (millions)")
ax.set_ylabel("Test Accuracy (%)")
ax.set_xlim(left=0.0)
ax.set_ylim(86.0, 93.8)
ax.legend(loc="lower right")
ax.tick_params(which="both", direction="out")

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(os.path.join(PLOT_DIR, f"fig2_pareto_frontier.{ext}"),
                bbox_inches="tight")
plt.show()
print("✓ Figure 2 saved.")

✓ Figure 2 saved.


In [26]:
# ── Figure 3: Per-Layer λ_max vs. Final Rank Allocation ──────────────────────
# Two-panel figure: top = Phase 2 curvature (λ_max), bottom = Phase 4 final ranks
# Shows the direct mapping from curvature signal to rank decision
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np, os

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "legend.fontsize": 8.5,
    "legend.frameon": True,
    "legend.edgecolor": "#cccccc",
    "legend.framealpha": 0.9,
})

layers = list(range(12))
x = np.arange(12)

# Per-layer λ_max: max(query, value) at r=16
lmax_per_layer = np.array([
    max(PHASE2_LAYER_LMAX_R16.get((i, "query"), 0.0),
        PHASE2_LAYER_LMAX_R16.get((i, "value"), 0.0))
    for i in layers
])

# Final ranks for all three runs
runs_info = {
    "PL_C1_tau75pct": (r"$\tau$=75th pct (90.61%)", "#6baed6"),
    "PL_C2_tau50pct": (r"$\tau$=50th pct (90.72%)", "#2171b5"),
    "PL_C3_PerLayer": (r"$\tau$=per-layer (90.85%)", "#08306b"),
}

fig = plt.figure(figsize=(10, 6.5))
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 1.8], hspace=0.45)

# ── Top panel: curvature bar (λ_max per layer) ───────────────────────────────
ax_top = fig.add_subplot(gs[0])
ax_top.bar(x, lmax_per_layer, color="#bdbdbd", edgecolor="#636363",
           linewidth=0.6, width=0.6)
ax_top.set_yscale("log")
ax_top.set_ylabel(r"$\lambda_{\max}$ (log scale)")
ax_top.set_xticks(x)
ax_top.set_xticklabels([f"L{i}" for i in layers])
ax_top.tick_params(axis="x", bottom=False)
ax_top.set_xlim(-0.6, 11.6)

# Annotate L11 query as the outlier
ax_top.annotate(
    r"L11 query: $\lambda$=1.23$\times10^{-2}$",
    xy=(11, lmax_per_layer[11]),
    xytext=(8.5, lmax_per_layer[11] * 1.8),
    fontsize=7.5, color="#333333",
    arrowprops=dict(arrowstyle="-", color="#999999", lw=0.8),
)
ax_top.spines["top"].set_visible(False)
ax_top.spines["right"].set_visible(False)

# ── Bottom panel: grouped bar — final ranks per layer, one bar per run ────────
ax_bot = fig.add_subplot(gs[1])
n_runs = sum(1 for rn in runs_info if rn in phase4_results)
bar_width = 0.22
offsets = np.linspace(-(n_runs - 1) * bar_width / 2,
                       (n_runs - 1) * bar_width / 2, n_runs)

for idx, (rn, (label, color)) in enumerate(runs_info.items()):
    if rn not in phase4_results:
        continue
    fr = phase4_results[rn]["final_ranks"]
    ranks = np.array([int(fr[str(i)]) for i in layers])
    ax_bot.bar(x + offsets[idx], ranks, width=bar_width,
               color=color, edgecolor="white", linewidth=0.5,
               label=label, alpha=0.88)

# r_init reference line
ax_bot.axhline(P4["r_init"], color="#aaaaaa", linestyle="--",
               linewidth=1.0, label=f"$r_{{\\rm init}}$ = {P4['r_init']}")

ax_bot.set_xlabel("Transformer Block Index")
ax_bot.set_ylabel("Final LoRA Rank")
ax_bot.set_xticks(x)
ax_bot.set_xticklabels([f"L{i}" for i in layers])
ax_bot.set_ylim(0, P4["r_max"] + 10)
ax_bot.set_yticks([0, 16, 30, 52, 64])
ax_bot.set_xlim(-0.6, 11.6)
ax_bot.legend(loc="upper left", ncol=2)
ax_bot.spines["top"].set_visible(False)
ax_bot.spines["right"].set_visible(False)

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(os.path.join(PLOT_DIR, f"fig3_rank_allocation.{ext}"),
                bbox_inches="tight")
plt.show()
print("✓ Figure 3 saved.")

✓ Figure 3 saved.
